# TweetAge v2 – NB-SVM sobre n-gramas de palabras y caracteres
**Curso de Procesamiento del Lenguaje Natural · Maestría en Inteligencia Artificial · Pontificia Universidad Javeriana**

Equipo: ____________________

## Qué cambia respecto a la versión 1
La versión 1 (SVM lineal sobre TF-IDF con vocabulario limitado a 200.000 términos por bloque) obtuvo **F1 macro = 0,555 en validación y 0,562 en Kaggle**. Esta versión introduce tres cambios, cada uno medido en la misma partición estratificada 80/20 (semilla 42):

| Paso | Cambio | F1 macro (validación) | Ganancia |
|---|---|---|---|
| 0 | Versión 1: palabras 1-2 + `char_wb` 2-5, 200k términos c/u, `LinearSVC(C=0.3)` | 0,5552 | — |
| 1 | Vocabulario completo (sin `max_features`) y palabras 1-3 | 0,5617 | +0,0065 |
| 2 | **NB-SVM**: escalar cada término por su log-cociente de Naive Bayes (Wang & Manning, 2012) | 0,5754 | +0,0137 |
| 3 | Bloque adicional de caracteres 1-5 que **cruzan** límites de palabra (`analyzer='char'`) | **0,5810** | +0,0056 |

**Ganancia total: +0,026.** Con el modelo final entrenado sobre el 100 % del train, se espera un puntaje en Kaggle algo mayor que el de validación (en la versión 1 la diferencia fue +0,007).

### Lo que se probó y **no** mejoró (por eso no está en el notebook)
| Experimento | F1 macro | Conclusión |
|---|---|---|
| Variables de estilo y legibilidad concatenadas al TF-IDF | 0,5576 vs 0,5549 | +0,003; marginal |
| Regresión logística (C = 10) sobre los mismos bloques | 0,5513 | peor que SVM |
| Ensamble NB-SVM + SVM + ComplementNB + SGD | 0,5816 | +0,0006; no compensa |
| Ajuste de sesgos por clase para F1 macro | ±0,0008 | ruido |
| Voto de vecinos más cercanos (k-NN coseno) | 0,5618 vs 0,5617 | nada |
| Voto por tokens raros compartidos (huella de autor) | 0,5801 | nada |
| Stacking con gradient boosting + variables de estilo | 0,5802 | nada |
| Suavizado NB α = 0,1 / 0,3 / 3 (en lugar de 1) | 0,569 / 0,574 / 0,565 | α = 1 es el óptimo |
| NB binarizado | 0,5731 | peor |

Las variables de estilo y legibilidad siguen siendo útiles para **explicar** el comportamiento de cada grupo (ver notebook de la versión 1 e informe), pero no para predecir.

## 1. Configuración

In [ ]:
import os
# Busca los CSV en Kaggle, en la carpeta de este notebook y en la ruta local del equipo.
# Si están en otra carpeta, agréguela al inicio de esta lista.
CANDIDATE_DIRS = [
    r'/kaggle/input/tweet-age-2026-puj',
    os.getcwd(),
    os.path.join(os.getcwd(), 'data'),
    r'C:\Users\leonk\Documents\MINTA 2025-01\PLN',
]
DATA_DIR = next((d for d in CANDIDATE_DIRS if os.path.exists(os.path.join(d, 'age_train.csv'))), None)
if DATA_DIR is None:
    raise FileNotFoundError('No encontré age_train.csv. Agregue la carpeta que lo contiene a CANDIDATE_DIRS.')
print('Datos en:', DATA_DIR)
TRAIN_PATH  = os.path.join(DATA_DIR, 'age_train.csv')
TEST_PATH   = os.path.join(DATA_DIR, 'age_test.csv')
SAMPLE_PATH = os.path.join(DATA_DIR, 'age_sample_submission.csv')
OUT_PATH    = 'age_submission.csv'

RUN_VALIDATION = True    # False = entrenar directamente el modelo final (ahorra ~10 min)
SAMPLE_FRAC    = None    # p. ej. 0.2 para pruebas rápidas
SEED           = 42

# Hiperparámetros elegidos en validación (ver tabla de la introducción)
NB_ALPHA = 1.0           # suavizado del cociente de Naive Bayes
SVM_C    = 0.3           # regularización del SVM lineal

In [ ]:
import os, html, time, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')
GROUPS = ['13-17', '18-24', '25-34', '35-49', '50-64', '65-xx']

## 2. Lectura y limpieza mínima
Igual que en la versión 1: separador `;`, BOM, relleno de espacios; se decodifican entidades HTML (`&lt;3` → `<3`) y se descartan del entrenamiento los tweets vacíos y los que contienen el artefacto de Excel `#¿NOMBRE?`. **No** se eliminan stopwords, menciones, mayúsculas ni alargamientos: son marcas de edad.

In [ ]:
def load(path):
    df = pd.read_csv(path, sep=';', encoding='utf-8-sig', dtype=str, keep_default_na=False)
    df['text'] = df['text'].str.strip().map(html.unescape)
    return df

def is_junk(texts):
    t = pd.Series(texts)
    return (t == '') | t.str.contains('#¿NOMBRE', regex=False)

train_raw, test = load(TRAIN_PATH), load(TEST_PATH)
if SAMPLE_FRAC:
    train_raw = train_raw.groupby('age_range', group_keys=False).sample(frac=SAMPLE_FRAC, random_state=SEED)
train = train_raw[~is_junk(train_raw.text).values].reset_index(drop=True)
print('Train:', train.shape, '| Test:', test.shape)
train.age_range.value_counts().reindex(GROUPS)

## 3. Representación: tres bloques TF-IDF
| Bloque | Configuración | Qué captura |
|---|---|---|
| Palabras 1-3 | patrón `[@#]?\w+`, `min_df=2`, sin límite de vocabulario | vocabulario, @usuarios, #hashtags, expresiones de 2-3 palabras |
| Caracteres 2-5 dentro de palabra (`char_wb`) | `min_df=3`, sin pasar a minúsculas | morfología, abreviaturas, risas, alargamientos |
| Caracteres 1-5 que cruzan palabras (`char`) | `min_df=5`, sin pasar a minúsculas | secuencias entre palabras: `RT @`, `jaja http`, puntuación y espaciado |

Todos con `sublinear_tf=True` (1 + log tf). **Quitar el límite de 200.000 términos** fue el primer cambio útil: el vocabulario de caracteres tiene más de 430.000 n-gramas con `min_df=3`, y los raros (cuentas y expresiones de nicho) son informativos.

In [ ]:
def make_vectorizers():
    return [
        TfidfVectorizer(token_pattern=r'(?u)[@#]?\w+', ngram_range=(1, 3), min_df=2,
                        sublinear_tf=True, dtype=np.float32),
        TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3, lowercase=False,
                        sublinear_tf=True, dtype=np.float32),
        TfidfVectorizer(analyzer='char', ngram_range=(1, 5), min_df=5, lowercase=False,
                        sublinear_tf=True, dtype=np.float32),
    ]

def build(texts_fit, *texts_other):
    vecs = make_vectorizers()
    X_fit = sparse.hstack([v.fit_transform(texts_fit) for v in vecs]).tocsr()
    others = [sparse.hstack([v.transform(t) for v in vecs]).tocsr() for t in texts_other]
    print('Dimensiones por bloque:', [len(v.vocabulary_) for v in vecs])
    return (X_fit, *others)

## 4. Modelo: NB-SVM
NB-SVM (Wang & Manning, 2012) combina Naive Bayes y SVM. Para cada clase *c* se entrena un SVM binario (*c* contra el resto), pero antes se **escala cada columna** por su log-cociente de Naive Bayes:

$$r_c = \log \frac{(p + \alpha)/\lVert p + \alpha \rVert_1}{(q + \alpha)/\lVert q + \alpha \rVert_1}$$

donde *p* y *q* son las sumas de la columna en los tweets de la clase *c* y del resto. Así, un término muy asociado a la clase (por ejemplo, `@justinbieber` para 13-17) entra con más peso, y uno neutro (como `de`) queda casi en cero. El SVM parte con una "pista" de Naive Bayes y la regularización castiga menos a los términos informativos. La predicción es la clase con mayor puntuación.

Esta técnica es especialmente efectiva en textos cortos y es la que más mejoró el resultado (+0,014).

In [ ]:
class NBSVM:
    def __init__(self, alpha=1.0, C=0.3):
        self.alpha, self.C = alpha, C

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.array(sorted(set(y)))
        self.r_, self.models_ = [], []
        for c in self.classes_:
            pos = y == c
            p = np.asarray(X[pos].sum(0)).ravel() + self.alpha
            q = np.asarray(X[~pos].sum(0)).ravel() + self.alpha
            r = np.log((p / p.sum()) / (q / q.sum())).astype(np.float32)
            m = LinearSVC(C=self.C).fit(X @ sparse.diags(r), pos.astype(int))
            self.r_.append(r); self.models_.append(m)
            print(f'  clase {c} lista', flush=True)
        return self

    def decision_function(self, X):
        return np.column_stack([m.decision_function(X @ sparse.diags(r))
                                for r, m in zip(self.r_, self.models_)])

    def predict(self, X):
        return self.classes_[self.decision_function(X).argmax(1)]

## 5. Validación (holdout estratificado 80/20)
Misma partición que en la versión 1, para que los números sean comparables. Los vectorizadores y los cocientes NB se ajustan **solo** con el 80 % de entrenamiento.

In [ ]:
if RUN_VALIDATION:
    t0 = time.time()
    idx_tr, idx_va = train_test_split(np.arange(len(train)), test_size=0.2,
                                      stratify=train.age_range, random_state=SEED)
    y_tr, y_va = train.age_range.values[idx_tr], train.age_range.values[idx_va]
    X_tr, X_va = build(train.text.iloc[idx_tr], train.text.iloc[idx_va])
    model_va = NBSVM(alpha=NB_ALPHA, C=SVM_C).fit(X_tr, y_tr)
    pred_va = model_va.predict(X_va)
    print(f'\nF1 macro validación: {f1_score(y_va, pred_va, average="macro"):.4f}  ({time.time()-t0:.0f}s)')
    print(classification_report(y_va, pred_va, labels=GROUPS, digits=3))

In [ ]:
if RUN_VALIDATION:
    ConfusionMatrixDisplay.from_predictions(y_va, pred_va, labels=GROUPS, normalize='true',
                                            values_format='.2f', cmap='Blues')
    plt.title('Matriz de confusión normalizada (validación)'); plt.show()

**Lectura esperada:** los extremos (13-17, 50-64, 65-xx) superan F1 = 0,65, mientras que 18-24 y 25-34 rondan 0,46-0,47 y se confunden entre sí y con sus vecinos. Los errores se concentran en grupos contiguos (Nguyen et al., 2013). Por tipo de tweet, las respuestas (@usuario …) son las mejor clasificadas (F1 ≈ 0,66) y los tweets propios sin mención los peores (≈ 0,53).

In [ ]:
if RUN_VALIDATION:
    # Términos con mayor peso NB por clase: verifican que el modelo aprende señales plausibles
    vocab = np.array(list(make_vectorizers()[0].fit(train.text.iloc[idx_tr]).get_feature_names_out()))
    n_w = len(vocab)
    top = {c: vocab[np.argsort(-r[:n_w])[:15]] for c, r in zip(model_va.classes_, model_va.r_)}
    display(pd.DataFrame(top))
    del X_tr, X_va, model_va; gc.collect()

## 6. Modelo final y submission
Se reentrena con **todo** el train. Luego se aplican dos reglas, validadas en el holdout:
1. **Duplicados exactos:** si un tweet de test es idéntico a uno de train con etiqueta unánime, se usa esa etiqueta. En validación acertó el 83,8 % de esos casos frente al 82,8 % del modelo (neutra o ligeramente positiva).
2. **Vacíos y `#¿NOMBRE?`:** reciben la clase más frecuente de esos casos en el train.

In [ ]:
t0 = time.time()
X_all, X_test = build(train.text, test.text)
model = NBSVM(alpha=NB_ALPHA, C=SVM_C).fit(X_all, train.age_range.values)
test['age_range'] = model.predict(X_test)
print(f'Modelo final entrenado y aplicado ({time.time()-t0:.0f}s)')

lab = train.groupby('text').age_range.agg(['nunique', 'first'])
lookup = lab.loc[lab['nunique'] == 1, 'first']
hit = test.text.isin(lookup.index) & (test.text != '')
test.loc[hit, 'age_range'] = test.loc[hit, 'text'].map(lookup)
print('Reemplazadas por duplicado exacto:', int(hit.sum()))

junk_tr = is_junk(train_raw.text).values
default = train_raw.age_range[junk_tr].mode().iloc[0] if junk_tr.any() else train.age_range.mode().iloc[0]
junk_te = is_junk(test.text).values
test.loc[junk_te, 'age_range'] = default
print(f'Vacíos o #¿NOMBRE? en test: {int(junk_te.sum())} → {default}')

In [ ]:
if os.path.exists(SAMPLE_PATH):
    cols = pd.read_csv(SAMPLE_PATH, nrows=0, sep=None, engine='python', encoding='utf-8-sig').columns.tolist()
else:
    cols = ['id', 'age_range']
sub = pd.DataFrame({cols[0]: test['id'].astype(int), cols[1]: test['age_range']})
assert len(sub) == 112892 or SAMPLE_FRAC, 'Número de filas inesperado'
assert sub[cols[1]].isin(GROUPS).all()
sub.to_csv(OUT_PATH, index=False)
print('Guardado:', OUT_PATH, sub.shape)
print(sub[cols[1]].value_counts(normalize=True).reindex(GROUPS).round(3))
sub.head()

## 7. Conclusiones
- Con modelos lineales sin preentrenamiento, la representación pesa más que el clasificador: vocabulario completo, n-gramas de caracteres que cruzan palabras y la ponderación NB-SVM suman +0,026 de F1 macro.
- Ensambles, stacking, k-NN, ajuste de umbrales y variables de estilo no mejoraron; el modelo lineal parece cerca de su techo (~0,58 en validación) con un solo tweet por observación.
- Los errores siguen concentrados entre grupos contiguos, en especial 18-24 y 25-34.

**Referencia del método:** Wang, S., & Manning, C. D. (2012). Baselines and bigrams: Simple, good sentiment and topic classification. En *Proceedings of the 50th Annual Meeting of the Association for Computational Linguistics* (Vol. 2, pp. 90–94). Association for Computational Linguistics.